<a href="https://colab.research.google.com/github/MHamza-Ahmad/Flyrank-Internship/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MHamza-Ahmad/Flyrank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
import pandas as pd
import numpy as np

# Try to load the .jsonl file, fallback to synthetic data if missing so Run All passes
file_path = 'data/NOBLE-cs-300.jsonl' # Updated file path to the downloaded JSONL file
try:
    df = pd.read_json(file_path, lines=True) # Changed to read_json for JSONL format
    print(f"Loaded '{file_path}' successfully.")
    print("Available columns in the DataFrame:", df.columns.tolist())

    # Extract date from the 'id' column and convert to datetime
    # The 'id' column format is typically 'sample_YYYYMMDD_HHMMSS_XXXXXX'
    df['date'] = pd.to_datetime(df['id'].str.split('_').str[1], format='%Y%m%d')

except FileNotFoundError:
    print(f"File '{file_path}' not found. Generating sample data to verify code logic...")
    dates = pd.date_range(start='2023-01-01', end='2023-12-31')
    df = pd.DataFrame({
        'date': dates,
        'product_id': 'PROD-A',
        'store_id': 'STORE-1',
        'sales_volume': np.random.randint(10, 100, size=len(dates)),
        'price': 19.99,
        'promotion_active': np.random.choice([0, 1], size=len(dates)),
        'transaction_id': range(1000, 1000 + len(dates))
    })

# Verify Unit and Window
print(f"Row count: {len(df)}")
print(f"Date Window: {df['date'].min()} to {df['date'].max()}")

Loaded 'data/NOBLE-cs-300.jsonl' successfully.
Available columns in the DataFrame: ['labels_v', 'scenario', 'signals', 'tags', 'context_state', 'qa', 'model_response_A', 'model_response_B', 'model_response_C', 'diff_notes', 'qa_ok', 'qa_fail_reasons', 'id']
Row count: 300
Date Window: 2025-12-23 00:00:00 to 2025-12-23 00:00:00


In [ ]:
# Install the Hugging Face Hub library
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import hf_hub_download

repo_id = "nowsika/NOBLE-b2b-saas-cs-starter-300"
filename = "data/NOBLE-cs-300.jsonl" # Updated to the correct path/filename

try:
    # Download the .jsonl file, specifying repo_type='dataset'
    downloaded_file_path = hf_hub_download(repo_id=repo_id, filename=filename, local_dir='.', local_dir_use_symlinks=False, repo_type='dataset')
    print(f"Successfully downloaded {filename} to {downloaded_file_path}")
except Exception as e:
    print(f"Error downloading {filename}: {e}")
    print("Please ensure the dataset is public or you are authenticated if it's private. You might need to use `huggingface_hub.login()` if it's private.")

Successfully downloaded data/NOBLE-cs-300.jsonl to /content/data/NOBLE-cs-300.jsonl


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


In [ ]:
from huggingface_hub import HfApi

api = HfApi()

repo_id = "nowsika/NOBLE-b2b-saas-cs-starter-300"

try:
    # List files in the dataset
    files_in_repo = api.list_repo_files(repo_id=repo_id, repo_type='dataset')
    print(f"Files found in the dataset '{repo_id}':")
    for file in files_in_repo:
        print(f"- {file}")
    print("Please identify the correct filename from the list above and let me know.")
except Exception as e:
    print(f"Error listing files for {repo_id}: {e}")
    print("Please ensure the dataset ID is correct and accessible.")

Files found in the dataset 'nowsika/NOBLE-b2b-saas-cs-starter-300':
- .gitattributes
- README.md
- SCENARIO_INDEX.md
- data/NOBLE-cs-300.jsonl
Please identify the correct filename from the list above and let me know.


The `starter.csv` file has been downloaded. Now, you can re-run cell `vAxwjrkyx7a1` to load this downloaded file into the DataFrame.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
Feature (inputs): price, promotion_active. These drive the predictions.

Label (target): sales_volume. This is the exact metric the model is trying to forecast.

Context (metadata): date, product_id, store_id. These define the boundaries and grain of the row.

Excluded: transaction_id or individual customer names. Why: Forecasting happens at an aggregated daily level to support broad business decisions; individual transaction grains introduce noise and risk overfitting.

In [ ]:












features = ['scenario', 'signals', 'tags', 'context_state', 'model_response_A', 'model_response_B', 'model_response_C', 'diff_notes']
label = 'qa_ok'
context = ['id', 'date'] # Now 'date' is available from 'id'
excluded = ['labels_v', 'qa', 'qa_fail_reasons'] # Exclude other columns not used in features, label, or context

# Convert 'qa_ok' to numerical for modeling purposes
if label in df.columns:
    df[label] = df[label].map({'YES': 1, 'NO': 0}).fillna(0) # Convert 'YES' to 1, 'NO' to 0, fill NaNs with 0.

# Verify the retained columns exist in the dataframe
expected_columns = features + [label] + context
missing_cols = [col for col in expected_columns if col not in df.columns]

assert len(missing_cols) == 0, f"Missing columns in dataset: {missing_cols}"
print("All contract fields are present in the dataset.")

All contract fields are present in the dataset.


In [ ]:
display(df.head())

,labels_v,scenario,signals,tags,context_state,qa,model_response_A,model_response_B,model_response_C,diff_notes,qa_ok,qa_fail_reasons,id,date
0,3.2.1,I was charged twice for the same invoice - can...,"{'shadow': {'archetype': 'Greed', 'subtype': '...","[b2b, customer_support, saas, billing, invoice...","{'risk': {'self_harm': 'LOW', 'violence': 'LOW...","{'metaphor_domain': 'ARCH', 'language': 'en', ...",{'raw_essay': 'Thanks for flagging this. A dou...,{'reflection': 'We can take ownership of this ...,{'reflection': 'We can work on this today. Fir...,{'A_intent': 'A aims to reassure the customer ...,1,[],sample_20251223_045659_128930,2025-12-23
1,3.2.1,Our card was charged but the invoice still sho...,"{'shadow': {'archetype': 'Greed', 'subtype': '...","[b2b, customer_support, saas, billing, invoice...","{'risk': {'self_harm': 'LOW', 'violence': 'LOW...","{'metaphor_domain': 'ARCH', 'language': 'en', ...",{'raw_essay': 'Thanks for flagging this. When ...,{'reflection': 'If your card shows a charge bu...,{'reflection': 'If your card shows a charge bu...,{'A_intent': 'A aims to reassure the customer ...,1,[],sample_20251223_045723_061094,2025-12-23
2,3.2.1,We need an invoice with our company legal name...,"{'shadow': {'archetype': 'Greed', 'subtype': '...","[b2b, customer_support, saas, billing, invoice...","{'risk': {'self_harm': 'LOW', 'violence': 'LOW...","{'metaphor_domain': 'ARCH', 'language': 'en', ...",{'raw_essay': 'Absolutely — we can get your in...,{'reflection': 'Yes — you can update the legal...,{'reflection': 'Yes — you can update the legal...,{'A_intent': 'A reassures the customer and exp...,1,[],sample_20251223_045746_070774,2025-12-23
3,3.2.1,Can you split this invoice across two cost cen...,"{'shadow': {'archetype': 'Greed', 'subtype': '...","[b2b, customer_support, saas, billing, invoice...","{'risk': {'self_harm': 'LOW', 'violence': 'LOW...","{'metaphor_domain': 'ARCH', 'language': 'en', ...",{'raw_essay': 'Yes—we can usually restructure ...,{'reflection': 'Yes—we can split the invoice a...,{'reflection': 'Yes—we can split the invoice a...,{'A_intent': 'Response A reassures the custome...,1,[],sample_20251223_045807_028873,2025-12-23
4,3.2.1,We accidentally upgraded seats - can you prora...,"{'shadow': {'archetype': 'Greed', 'subtype': '...","[b2b, customer_support, saas, billing, invoice...","{'risk': {'self_harm': 'LOW', 'violence': 'LOW...","{'metaphor_domain': 'ARCH', 'language': 'en', ...",{'raw_essay': 'Yes—we can look at prorating an...,{'reflection': 'We can review the seat upgrade...,{'reflection': 'We can review the accidental s...,{'A_intent': 'A aims to reassure the customer ...,1,[],sample_20251223_045827_651023,2025-12-23


In [ ]:
print(df.columns.tolist())

['labels_v', 'scenario', 'signals', 'tags', 'context_state', 'qa', 'model_response_A', 'model_response_B', 'model_response_C', 'diff_notes', 'qa_ok', 'qa_fail_reasons', 'id', 'date']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# 1. Total row count
print(f"Total rows: {len(df)}")

# 2. Missing values in critical fields
missing_labels = df[label].isnull().sum()
print(f"Missing '{label}' values: {missing_labels}")

# 3. Verify grain: combination of 'id' and 'date' should be unique based on the new context
duplicates = df.duplicated(subset=context).sum()
print(f"Duplicate rows based on context grain ({', '.join(context)}): {duplicates}")

# 4. Feature completeness
missing_features = df[features].isnull().sum().to_dict()
print(f"Missing values in features: {missing_features}")

Total rows: 300
Missing 'qa_ok' values: 0
Duplicate rows based on context grain (id, date): 0
Missing values in features: {'scenario': 0, 'signals': 0, 'tags': 0, 'context_state': 0, 'model_response_A': 0, 'model_response_B': 0, 'model_response_C': 0, 'diff_notes': 0}


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
This data is strictly observational and directional. It captures historical sales patterns but can never tell you the underlying human reason a customer bought the product (no causal proof). Furthermore, because the time window is bounded, the model cannot account for completely novel market shocks—like a sudden global supply chain disruption or a brand-new competitor launching today

In [ ]:
latest_date = df['date'].max()
print(f"Data limit acknowledged: Model has no visibility into events after {latest_date}.")

Data limit acknowledged: Model has no visibility into events after 2025-12-23 00:00:00.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.